# WoodelfHD, m=100 background: WoodelfHD on GPU (notebook 3)

Runs `benchmarks/woodelfhd_m100_gpu_experiment.py`: WoodelfHD on a GPU, on the **same models, explained rows, background rows and tree counts** as WoodelfHD on CPU in notebook 1. Only the GPU method runs here.

**Runtime:** T4 GPU with High-RAM (Runtime → Change runtime type → T4 GPU, High-RAM on).

**Before you start:** WoodelfHD must have finished in notebook 1, because the GPU cells load the models it trained into `models_main/` (step 6 checks this and stops otherwise). Notebook 1 may still be running its other methods; this notebook only reads its models.

**Estimated time: about 3–3.5 h.** GPU compute is about 3 h, estimated from the m=100 CPU times and the GPU/CPU ratios of the paper's Tables 3 and 5. About 2 h of it is California Housing, where the GPU is slower than the CPU (as in Table 5). Setup and data loading add about 15 min.

### If the runtime crashes or disconnects
Reconnect and **run all cells from the top**. Results are saved to Drive after every finished cell (`results/woodelf_hd_gpu.json`); step 5 restores them and the run skips those cells. Only the cell that was running when the runtime died is lost.

### Drive layout (shared with notebook 1)
```
MyDrive/treebranchmarks/woodelfhd_m100/
  results/       woodelf_hd_gpu.json (written here), notebook 1 and 2 results (only read)
  datasets/      prepared by notebook 1 (only read)
  models_main/   notebook 1's models (only read)
  woodelfhd_m100_gpu_experiment.html   the report (step 10), CPU and GPU rows side by side
```

In [ ]:
# ── Step 1: Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Step 2: Configuration ───────────────────────────────────────────────────
import os, pathlib

DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/treebranchmarks/woodelfhd_m100')   # notebook 1's folder
METHOD     = 'woodelf_hd_gpu'

REPO_URL    = 'https://github.com/ron-wettenstein/TreeBranchMarks.git'
REPO_BRANCH = 'feature/m100_woodelfhd_gpu'
REPO        = pathlib.Path('/content/treebranchmarks')
TOOLS       = REPO / 'colab' / 'woodelfhd_m100_gpu_tools.py'   # restore / check-models / progress / report

# Optional: the woodelf_explainer version notebook 1 ran with (its pip output shows it),
# e.g. '0.4.8'. Leave empty to use the version pip installs.
WOODELF_VERSION = ''

RESULTS_DRIVE = DRIVE_ROOT / 'results'
for p in (DRIVE_ROOT / 'datasets', DRIVE_ROOT / 'models_main', RESULTS_DRIVE):
    if not p.exists():
        raise FileNotFoundError(f'{p} not found. Run notebook 1 first.')
print('Drive folder:', DRIVE_ROOT)

In [ ]:
# ── Step 3: Check the GPU and the RAM ───────────────────────────────────────
import subprocess
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
if not gpu:
    raise RuntimeError('No GPU found. Runtime -> Change runtime type -> T4 GPU.')
ram_gb = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1e9
print(f'GPU: {gpu} | RAM: {ram_gb:.0f} GB')
if 'T4' not in gpu:
    print('WARNING: not a T4. The paper reports T4 times.')
if ram_gb < 45:
    print('WARNING: High-RAM is off. Turn it on (Runtime -> Change runtime type).')

In [ ]:
# ── Step 4: Clone the repo and install ──────────────────────────────────────
if not REPO.exists():
    !git clone -q -b {REPO_BRANCH} {REPO_URL} {REPO}
if not (REPO / 'benchmarks' / 'woodelfhd_m100_gpu_experiment.py').exists() or not TOOLS.exists():
    raise FileNotFoundError(
        'The GPU experiment files are not in the cloned repo. Push them to GitHub '
        '(or set REPO_BRANCH), delete /content/treebranchmarks and rerun this cell.')

!pip install -q -e {REPO}
if WOODELF_VERSION:
    !pip install -q woodelf_explainer=={WOODELF_VERSION}
try:
    import cupy
except ImportError:
    !pip install -q cupy-cuda12x
    import cupy

import importlib.metadata as md
versions = {}
for pkg in ('woodelf_explainer', 'lightgbm', 'cupy-cuda12x', 'numpy'):
    try:
        versions[pkg] = md.version(pkg)
    except md.PackageNotFoundError:
        pass
versions['gpu'] = cupy.cuda.runtime.getDeviceProperties(0)['name'].decode()
print(versions)
(DRIVE_ROOT / 'woodelf_hd_gpu_environment.txt').write_text(str(versions) + '\n')

In [ ]:
# ── Step 5: Restore datasets, models and finished GPU results from Drive ────
!python {TOOLS} restore --drive-root "{DRIVE_ROOT}" --methods {METHOD} --models-dir models_main

In [ ]:
# ── Step 6: Check that notebook 1's models are all there ────────────────────
# Stops if any GPU cell would have to train its own model: the GPU cells must use
# the same model files as the CPU cells.
!python {TOOLS} check-models --repo {REPO}

In [ ]:
# ── Step 7: Progress so far ─────────────────────────────────────────────────
!python {TOOLS} progress --drive-root "{DRIVE_ROOT}" --methods {METHOD}

## Run

In [ ]:
# ── Step 8: Run the experiment (WoodelfHD GPU only) ─────────────────────────
# --result_location is a folder: woodelf_hd_gpu.json there is rewritten after every finished cell.
os.chdir(REPO)
!python -u -m benchmarks.woodelfhd_m100_gpu_experiment --method {METHOD} --result_location "{RESULTS_DRIVE}"

In [ ]:
# ── Step 9: Progress after the run ──────────────────────────────────────────
!python {TOOLS} progress --drive-root "{DRIVE_ROOT}" --methods {METHOD}

## Report (after notebooks 1 and 2 have finished too)

Builds one HTML report with the CPU rows (notebooks 1 and 2) and the GPU rows (this notebook), from the result files on Drive. Nothing is run, trained or downloaded.
If any cell of any method is unfinished, it stops and lists those cells instead.

On a fresh runtime, run steps 1–4 first, then the cells below.

In [ ]:
# ── Step 10: Build the report ───────────────────────────────────────────────
!python {TOOLS} report --drive-root "{DRIVE_ROOT}"

In [ ]:
# ── Step 11: Download the report ────────────────────────────────────────────
from google.colab import files
files.download(str(DRIVE_ROOT / 'woodelfhd_m100_gpu_experiment.html'))